# Import Modules

In [20]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from utils import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

In [28]:
# 3 Datasets
df_chd_heart_stroke_intersection = get_unscaled_combined("chd-heart-stroke-intersection")
df_chd_heart_stroke_union = get_unscaled_combined("chd-heart-stroke-union")
df_framingham_heart_stroke_intersection = get_unscaled_combined("framingham-heart-stroke-intersection")
df_framingham_heart_stroke_union = get_unscaled_combined("framingham-heart-stroke-union")

# 2 Datasets
df_chd_stroke_intersection = get_unscaled_combined("chd-stroke-intersection")
df_chd_stroke_union = get_unscaled_combined("chd-stroke-union")
df_framingham_heart_intersection = get_unscaled_combined("framingham-heart-intersection")
df_framingham_heart_union = get_unscaled_combined("framingham-heart-union")

In [29]:
union_datasets = {
    "framingham_heart_union": df_framingham_heart_union,
    "chd_stroke_union": df_chd_stroke_union,
    "chd_heart_stroke_union": df_chd_heart_stroke_union,
    "framingham_heart_stroke_union": df_framingham_heart_stroke_union
}

intersection_datasets = {
    "chd_stroke_intersection": df_chd_stroke_intersection,
    "framingham_heart_intersection": df_framingham_heart_intersection,
    "chd_heart_stroke_intersection": df_chd_heart_stroke_intersection,
    "framingham_heart_stroke_intersection": df_framingham_heart_stroke_intersection
}

In [23]:
def check_missing_values(df, dataset_name="Dataset"):
    print(f"=== Missing Value Report: {dataset_name} ===")
    
    # 1. Check if ANY missing values exist overall
    has_missing = df.isnull().values.any()

    if has_missing:
        # 2. Count missing values for each column
        missing_counts = df.isnull().sum()
        
        # Filter to show only columns that actually have missing data
        missing_columns = missing_counts[missing_counts > 0]
        
        # 3. Calculate the percentage of missing data per column
        missing_percentages = (missing_columns / len(df)) * 100
        
        # Combine counts and percentages into a clean table
        summary_df = pd.DataFrame({
            'Missing Count': missing_columns,
            'Percentage (%)': missing_percentages.round(2)
        }).sort_values(by='Missing Count', ascending=False)
        
        print("Columns with missing values:")
        print(summary_df)
        print("\n")
    else:
        print("Great news! This dataset is completely full with no missing values.\n")


check_missing_values(df_framingham_heart_union, "Framingham-Heart Union")
check_missing_values(df_chd_stroke_union, "CHD-Stroke Union")
check_missing_values(df_chd_heart_stroke_union, "CHD-Heart-Stroke Union")
check_missing_values(df_framingham_heart_stroke_union, "Framingham-Heart-Stroke Union")
check_missing_values(df_chd_stroke_intersection, "CHD-Stroke Intersection")
check_missing_values(df_framingham_heart_intersection, "Framingham-Heart Intersection")
check_missing_values(df_chd_heart_stroke_intersection, "CHD-Heart-Stroke Intersection")
check_missing_values(df_framingham_heart_stroke_intersection, "Framingham-Heart-Stroke Intersection")

=== Missing Value Report: Framingham-Heart Union ===
Columns with missing values:
                     Missing Count  Percentage (%)
thal                          4240           93.35
mv                            4240           93.35
slope                         4240           93.35
oldpeak                       4240           93.35
angina                        4240           93.35
ecg                           4240           93.35
fbs                           4240           93.35
cp_type                       4240           93.35
smoking_status                 302            6.65
edu                            302            6.65
bmi                            302            6.65
dbp                            302            6.65
diabetes_status                302            6.65
hypertension_status            302            6.65
stroke_status                  302            6.65
bp_status                      302            6.65
num_cigs_per_day               302            6.65


# Data Preprocessing

In [24]:
def process_union(df, name, drop_threshold=0.75):
    df_processed = df.copy()
    
    # 1. DROP columns with extreme missing values (e.g., > 75% missing)
    missing_fractions = df_processed.isnull().mean()
    cols_to_drop = missing_fractions[missing_fractions > drop_threshold].index
    
    if len(cols_to_drop) > 0:
        print(f"⚠️ [{name}] Dropping columns (> {drop_threshold*100}% missing): {list(cols_to_drop)}")
        df_processed.drop(columns=cols_to_drop, inplace=True)

    numeric_cols = df_processed.select_dtypes(include=['number']).columns
    categorical_cols = df_processed.select_dtypes(exclude=['number']).columns

    # 2. IMPUTE the remaining columns
    if len(numeric_cols) > 0:
        num_imputer = SimpleImputer(strategy='most_frequent')
        df_processed[numeric_cols] = num_imputer.fit_transform(df_processed[numeric_cols])

    if len(categorical_cols) > 0:
        cat_imputer = SimpleImputer(strategy='most_frequent')
        df_processed[categorical_cols] = cat_imputer.fit_transform(df_processed[categorical_cols])

    # 3. SCALE the numeric features
    scaler = MinMaxScaler()
    if len(numeric_cols) > 0:
        df_processed[numeric_cols] = scaler.fit_transform(df_processed[numeric_cols])

    set_preprocessed_combined(df_processed, scaler, name)
    print(f"✅ Processed Union Dataset: {name}\n")

print("--- Processing Union Datasets ---")
for name, df in union_datasets.items():
    process_union(df, name)

--- Processing Union Datasets ---
⚠️ [framingham_heart_union] Dropping columns (> 75.0% missing): ['cp_type', 'fbs', 'ecg', 'angina', 'oldpeak', 'slope', 'mv', 'thal']
✅ Processed Union Dataset: framingham_heart_union

⚠️ [chd_stroke_union] Dropping columns (> 75.0% missing): ['sbp', 'tobacco', 'ldl', 'adiposity', 'famhist', 'typea', 'alcohol']
✅ Processed Union Dataset: chd_stroke_union

⚠️ [chd_heart_stroke_union] Dropping columns (> 75.0% missing): ['sbp', 'tobacco', 'ldl', 'adiposity', 'famhist', 'typea', 'alcohol', 'cp_type', 'chol', 'fbs', 'ecg', 'hr', 'angina', 'oldpeak', 'slope', 'mv', 'thal']
✅ Processed Union Dataset: chd_heart_stroke_union

⚠️ [framingham_heart_stroke_union] Dropping columns (> 75.0% missing): ['cp_type', 'fbs', 'ecg', 'angina', 'oldpeak', 'slope', 'mv', 'thal']
✅ Processed Union Dataset: framingham_heart_stroke_union



In [25]:
def process_intersection(df, name):
    df_processed = df.copy()
    numeric_cols = df_processed.select_dtypes(include=['number']).columns

    scaler = MinMaxScaler()
    if len(numeric_cols) > 0:
        df_processed[numeric_cols] = scaler.fit_transform(df_processed[numeric_cols])

    set_preprocessed_combined(df_processed, scaler, name)
    print(f"✅ Processed Intersection Dataset: {name}")
    
print("\n--- Processing Intersection Datasets ---")
for name, df in intersection_datasets.items():
    process_intersection(df, name)


--- Processing Intersection Datasets ---
✅ Processed Intersection Dataset: chd_stroke_intersection
✅ Processed Intersection Dataset: framingham_heart_intersection
✅ Processed Intersection Dataset: chd_heart_stroke_intersection
✅ Processed Intersection Dataset: framingham_heart_stroke_intersection
